In [1]:
# ==============================================================================
# 1. DEPENDENCIES & INDUSTRIAL ENVIRONMENT SETUP
# ==============================================================================
import numpy as np
import pandas as pd
import logging
import warnings
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Modeling Stack
from sklearn.linear_model import Ridge, LinearRegression, Lasso
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor

# Formatting and configurations
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("PCB_Thermal_Expansion")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ==============================================================================
# 2. EXPERIMENTAL LAB DATA SIMULATION (High-Fidelity Physics Base)
# ==============================================================================
def load_factory_pcb_dataset(n_samples=40):
    """
    Simulates physical measurements collected from a thermal moiré warpage system.
    Features mimic an 8-layer mixed glass-weave FR4 printed circuit board assembly.
    """
    logger.info(f"Ingesting raw manufacturing laboratory records. Sample count: {n_samples}")
    
    data = {
        'pcb_thickness_mm': np.random.uniform(0.8, 2.4, n_samples),
        'copper_density_top_pct': np.random.uniform(40.0, 85.0, n_samples),
        'copper_density_bot_pct': np.random.uniform(35.0, 80.0, n_samples),
        'resin_content_pct': np.random.uniform(42.0, 58.0, n_samples),
        'glass_transition_temp_C': np.random.uniform(135.0, 175.0, n_samples),
        'peak_reflow_temp_C': np.random.uniform(240.0, 260.0, n_samples),
        # Environmental noise variables to test feature robustness
        'cleanroom_humidity_pct': np.random.uniform(30.0, 55.0, n_samples),
        'sensor_calibration_drift': np.random.normal(0, 0.5, n_samples)
    }
    
    df = pd.DataFrame(data)
    
    # Target variable generation governed by true thermo-mechanical anomalies:
    # 1. Copper asymmetry creates an intrinsic bending moment (warpage)
    copper_asymmetry = np.abs(df['copper_density_top_pct'] - df['copper_density_bot_pct'])
    # 2. Severe non-linear Z-axis volumetric expansion occurs when peak reflow exceeds Tg
    thermal_excursion = np.maximum(0, df['peak_reflow_temp_C'] - df['glass_transition_temp_C'])
    
    # Target: Maximum out-of-plane displacement (Warpage) in Microns
    df['max_warpage_microns'] = (
        (copper_asymmetry * 2.1) + 
        (thermal_excursion ** 1.55) * 1.85 - 
        (df['pcb_thickness_mm'] * 28.5) + 
        (df['resin_content_pct'] * 0.4) + 
        np.random.normal(0, 2.0, n_samples) # Experimental measurement error
    )
    return df

# Load production dataframe
df_raw = load_factory_pcb_dataset(40)


2026-06-19 20:48:08,983 - INFO - Ingesting raw manufacturing laboratory records. Sample count: 40


In [2]:
# ==============================================================================
# 3. PRODUCTION EDA & PHYSICAL FEATURE ENGINEERING PIPELINE
# ==============================================================================
def production_data_engineering(df_in):
    """Performs strict data validation and injects physical engineering domain features."""
    logger.info("Executing exploratory data audit and feature injection loop...")
    df = df_in.copy()
    
    # Data Sanity Validation Checks
    assert (df['pcb_thickness_mm'] > 0).all(), "Critical Error: Negative board thickness detected."
    assert (df['peak_reflow_temp_C'] >= 0).all(), "Critical Error: Invalid negative thermal readings."
    
    # ─── Domain-Driven Feature Engineering ───
    # Feature 1: Copper Asymmetry (Asymmetrical layers drive uneven thermal expansion stresses)
    df['copper_imbalance_ratio'] = np.abs(df['copper_density_top_pct'] - df['copper_density_bot_pct'])
    
    # Feature 2: Thermal Excursion Metric (Delta above Tg dictates the polymer's viscoelastic state)
    df['thermal_delta_above_tg'] = np.maximum(0, df['peak_reflow_temp_C'] - df['glass_transition_temp_C'])
    
    # Feature 3: Structural Slenderness Ratio (Thinner boards buckle far easier under identical load)
    df['structural_slenderness'] = 1.0 / (df['pcb_thickness_mm'] ** 2)
    
    # Drop raw confounding variables that have been cleanly engineered
    features_to_drop = ['cleanroom_humidity_pct', 'sensor_calibration_drift']
    df = df.drop(columns=features_to_drop)
    
    logger.info(f"Feature engineering complete. Retained {df.shape[1] - 1} operational features.")
    return df

# Apply pipeline transformation
df_engineered = production_data_engineering(df_raw)

# Separate inputs and continuous target variable
X = df_engineered.drop(columns=['max_warpage_microns'])
y = df_engineered['max_warpage_microns']


2026-06-19 20:48:19,224 - INFO - Executing exploratory data audit and feature injection loop...
2026-06-19 20:48:19,227 - INFO - Feature engineering complete. Retained 9 operational features.


In [4]:
# ==============================================================================
# 4. OFFLINE STANDALONE PARAMETER OPTIMIZATION (Preventing LOOCV Bloat)
# ==============================================================================
def optimize_model_hyperparameters(X_train, y_train):
    """Trains rapid isolated cross-validated grids to freeze core hyperparameters."""
    logger.info("Running offline parameter optimization sweeps via 5-Fold Stratified-range CV...")
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    
    # Tune Ridge Alpha
    ridge_scores = {}
    for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
        pipe = Pipeline([('scaler', RobustScaler()), ('model', Ridge(alpha=alpha))])
        preds = cross_val_predict(pipe, X_train, y_train, cv=kf)
        ridge_scores[alpha] = root_mean_squared_error(y_train, preds)
    best_alpha = min(ridge_scores, key=ridge_scores.get)
    
    # Tune Gradient Boosting Depth and Estimator allocations
    gbr_scores = {}
    for depth in [3, 5, 8]:
        for lr in [0.01, 0.05, 0.1]:
            gbr = GradientBoostingRegressor(max_depth=depth, learning_rate=lr, n_estimators=70, random_state=RANDOM_STATE)
            preds = cross_val_predict(gbr, X_train, y_train, cv=kf)
            gbr_scores[(depth, lr)] = root_mean_squared_error(y_train, preds)
    best_depth, best_lr = min(gbr_scores, key=gbr_scores.get)
    
    logger.info(f"Hyperparameters Frozen -> Ridge Alpha: {best_alpha} | GBR Depth: {best_depth}, LR: {best_lr}")
    return best_alpha, best_depth, best_lr

# Retrieve optimized configurations
opt_alpha, opt_depth, opt_lr = optimize_model_hyperparameters(X, y)


2026-06-19 20:49:20,143 - INFO - Running offline parameter optimization sweeps via 5-Fold Stratified-range CV...
2026-06-19 20:49:21,727 - INFO - Hyperparameters Frozen -> Ridge Alpha: 0.1 | GBR Depth: 3, LR: 0.1


In [5]:
# ==============================================================================
# 5. PIPELINE DEFINTIONS & MODEL STACKING ARCHITECTURE
# ==============================================================================

# Pipeline Branch A: Linear Modelling via L2 Regularisation
ridge_pipeline = Pipeline([
    ('scaler', RobustScaler()), # Robust to potential manufacturing measurement outliers
    ('selector', SelectKBest(score_func=f_regression, k=5)),
    ('ridge', Ridge(alpha=opt_alpha, random_state=RANDOM_STATE))
])

# Pipeline Branch B: Non-Linear Gradient Boosting Tree Engine
gbr_pipeline = Pipeline([
    ('selector', SelectKBest(score_func=f_regression, k=5)),
    ('gbr', GradientBoostingRegressor(
        max_depth=opt_depth, 
        learning_rate=opt_lr, 
        n_estimators=70, 
        subsample=0.85, 
        random_state=RANDOM_STATE
    ))
])

# Master Architecture: Blended Stacking Regressor
stack_ensemble = StackingRegressor(
    estimators=[
        ('ridge_branch', ridge_pipeline),
        ('gbr_branch', gbr_pipeline)
    ],
    final_estimator=LinearRegression(fit_intercept=True), # Meta-learner evaluating spatial residuals
    cv=5, # 5-Fold Internal cross-validation split ensures clean out-of-fold blending predictions
    n_jobs=-1
)

# ==============================================================================
# 6. LEAVE-ONE-OUT CROSS-VALIDATION RIGOROUS TESTING LOOP
# ==============================================================================
loo = LeaveOneOut()
logger.info("Executing definitive LOOCV operational tournament...")

y_pred_ridge = cross_val_predict(ridge_pipeline, X, y, cv=loo, n_jobs=-1)
y_pred_gbr   = cross_val_predict(gbr_pipeline, X, y, cv=loo, n_jobs=-1)
y_pred_stack = cross_val_predict(stack_ensemble, X, y, cv=loo, n_jobs=-1)


2026-06-19 20:49:44,643 - INFO - Executing definitive LOOCV operational tournament...


In [6]:
# ==============================================================================
# 7. METRIC COMPILATION & MANUFACTURING SIGN-OFF REPORT
# ==============================================================================
def generate_audit_record(y_true, y_pred, name):
    return {
        'Model Pipeline': name,
        'RMSE (µm)': root_mean_squared_error(y_true, y_pred),
        'MAE (µm)': mean_absolute_error(y_true, y_pred),
        'R² Variance Explained': r2_score(y_true, y_pred)
    }

audit_log = [
    generate_audit_record(y, y_pred_ridge, "Baseline L2 Ridge Linear Pipeline"),
    generate_audit_record(y, y_pred_gbr, "Non-Linear Gradient Boosting Engine"),
    generate_audit_record(y, y_pred_stack, "Heterogeneous Stacking Ensemble (Blended)")
]

df_audit = pd.DataFrame(audit_log).set_index('Model Pipeline')

print("\n" + "="*72)
print("               PCB THERMO-MECHANICAL EXPANSION METRIC AUDIT")
print("="*72)
print(df_audit.round(4).to_string())
print("="*72)

# ==============================================================================
# 8. PRODUCTION METRIC DECONSTRUCTION (META-LEARNER BREAKDOWN)
# ==============================================================================
# Fit final production system to 100% of laboratory data for physical deployment
stack_ensemble.fit(X, y)
meta_learner = stack_ensemble.final_estimator_
weights = meta_learner.coef_

print("\n[Production Deployment Diagnostics]")
print(f" -> Meta-Learner Structural Intercept Adjust: {meta_learner.intercept_:.4f} µm")
print(f" -> Ridge Model Blending Coefficient       : {weights[0]:.4f}")
print(f" -> Gradient Boosting Blending Coefficient  : {weights[1]:.4f}")
print("-" * 72)

# Automated Engineering Interpretation
if weights[1] > weights[0]:
    print("Strategic Assessment: The system relies heavily on Non-Linear Gradient Boosting.\n"
          "Recommendation: Calibrate for severe viscoelastic polymer shearing past the Tg point.")
else:
    print("Strategic Assessment: The system relies heavily on Regularised Linear Ridge pathways.\n"
          "Recommendation: Mechanical behavior is dominated by consistent global bulk metal thermal coefficients.")
print("="*72)



               PCB THERMO-MECHANICAL EXPANSION METRIC AUDIT
                                           RMSE (µm)  MAE (µm)  R² Variance Explained
Model Pipeline                                                                       
Baseline L2 Ridge Linear Pipeline            27.5740   21.9683                 0.9966
Non-Linear Gradient Boosting Engine          68.1900   49.5320                 0.9793
Heterogeneous Stacking Ensemble (Blended)    28.8282   23.1058                 0.9963

[Production Deployment Diagnostics]
 -> Meta-Learner Structural Intercept Adjust: -18.6169 µm
 -> Ridge Model Blending Coefficient       : 0.8675
 -> Gradient Boosting Blending Coefficient  : 0.1412
------------------------------------------------------------------------
Strategic Assessment: The system relies heavily on Regularised Linear Ridge pathways.
Recommendation: Mechanical behavior is dominated by consistent global bulk metal thermal coefficients.
